# Chapter 6 — DataLoader

**Book alignment:** PyTorch From First Principles, Chapter 6

**Question this notebook isolates:** Is per-sample production cost (not the training step) what limits input throughput — i.e., does a slow producer measurably delay batch delivery while `shuffle`/`drop_last`/worker settings change only what (not whether) the loop receives?


In [ ]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
np.random.seed(0)


## 1. Batch contract: `drop_last` and `shuffle` change what the loop sees

100 samples, batch 16: `drop_last=False` must yield 7 batches (last has 4); `True` must yield 6 full ones; shuffling must reorder.


In [ ]:
X = torch.randn(100, 4)
y = torch.arange(100)
ds = TensorDataset(X, y)

keep = list(DataLoader(ds, batch_size=16, shuffle=False, drop_last=False))
drop = list(DataLoader(ds, batch_size=16, shuffle=False, drop_last=True))
shuf = list(DataLoader(ds, batch_size=16, shuffle=True, generator=torch.Generator().manual_seed(0)))
print(f"keep_last: {len(keep)} batches, last={tuple(keep[-1][0].shape)}")
print(f"drop_last: {len(drop)} batches, last={tuple(drop[-1][0].shape)}")
print(f"ordered head:   {keep[0][1][:4].tolist()}")
print(f"shuffled head:  {shuf[0][1][:4].tolist()}")


In [ ]:
assert len(keep) == 7 and tuple(keep[-1][0].shape) == (4, 4)
assert len(drop) == 6 and all(tuple(b[0].shape) == (16, 4) for b in drop)
assert sum(b[0].shape[0] for b in keep) == 100
assert sum(b[0].shape[0] for b in drop) == 96
assert shuf[0][1].tolist() != keep[0][1].tolist()
print("batch geometry and order pinned down")


## 2. Producer cost dominates delivery: slow vs precomputed source

Same batch count, same consumer: a dataset sleeping 8 ms/sample must take far longer to iterate than a precomputed tensor dataset (generous margin: 3x).


In [ ]:
class SlowDataset(torch.utils.data.Dataset):
    def __init__(self, n=48, dim=8, cost=0.008):
        self.n, self.dim, self.cost = n, dim, cost
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        time.sleep(self.cost)
        g = torch.Generator().manual_seed(i)
        return torch.randn(self.dim, generator=g), i % 2

def epoch_time(loader):
    t0 = time.perf_counter()
    k = 0
    for xb, yb in loader:
        k += 1
    return time.perf_counter() - t0, k

slow = DataLoader(SlowDataset(), batch_size=16, num_workers=0)
fast_ds = TensorDataset(torch.randn(48, 8), torch.arange(48) % 2)
fast = DataLoader(fast_ds, batch_size=16, num_workers=0)
t_slow, k_slow = epoch_time(slow)
t_fast, k_fast = epoch_time(fast)
print(f"slow: {t_slow:.3f}s over {k_slow} batches | fast: {t_fast:.4f}s over {k_fast} batches")


In [ ]:
assert k_slow == k_fast == 3
assert t_slow > 0.15, t_slow
assert t_slow > 3 * t_fast, (t_slow, t_fast)
print("delivery time lives in sample production, not the training step")


## 3. Workers change delivery, not contents; seeds make shuffling reproducible

Same seed must give the same shuffled order across loaders; `num_workers=2` must deliver identical values to `num_workers=0`.


In [ ]:
def order(seed):
    g = torch.Generator().manual_seed(seed)
    return torch.cat([b[1] for b in DataLoader(ds, batch_size=16, shuffle=True, generator=g)]).tolist()

o1, o2, o3 = order(123), order(123), order(999)
print(f"same seed identical: {o1 == o2}, different seed differs: {o1 != o3}")

w0 = [b for b in DataLoader(ds, batch_size=16, shuffle=False, num_workers=0)]
w2 = [b for b in DataLoader(ds, batch_size=16, shuffle=False, num_workers=2)]
same = all(torch.equal(a[0], b[0]) and torch.equal(a[1], b[1]) for a, b in zip(w0, w2))
print(f"workers=0 vs 2 identical contents: {same}")


In [ ]:
assert o1 == o2
assert o1 != o3
assert sorted(o1) == list(range(100))
assert same
print("parallelism relocates work; the batches are unchanged")


## What we earned

Split every step into *wait for batch* vs *work on batch*: slow delivery is a producer problem even when the training code looks guilty. Measure at the boundary, change one mechanism, and check whether the bottleneck moved or merely relocated.

Chapter 7 opens the batch and asks what the model actually sees: the representation the transforms produced.
